In [3]:
%%writefile streamlit_app.py

import streamlit as st
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
import os

# Load environment variables (for local development)
load_dotenv()

st.set_page_config(
    page_title="MoPhones Credit Portfolio",
    layout="wide",
    page_icon="🛡️"
)

st.title("🛡️ MoPhones Credit Portfolio Health Monitor")
st.markdown("### Real-time Credit Risk & Portfolio Dashboard")

# ==================== DATABASE CONNECTION ====================
# This works both locally and on Streamlit Cloud
try:
    conn = st.connection("mysql", type="sql")
    st.success("✅ Successfully connected to MySQL Database")
except Exception as e:
    st.error(f"Connection failed: {e}")
    st.stop()

# ==================== SIDEBAR ====================
with st.sidebar:
    st.success("Connected to Database")
    st.header("🔍 Filters")
    
    # We'll load data later when needed to avoid heavy loading

# ==================== KPI CARDS ====================
st.divider()

col1, col2, col3, col4 = st.columns(4)

with col1:
    kpi = conn.query("SELECT * FROM mart_portfolio_kpis LIMIT 1")
    st.metric("**Total Disbursed**", f"KES {kpi['total_disbursed'].iloc[0]:,}")

with col2:
    st.metric("**PAR30**", f"{kpi['PAR30'].iloc[0]}%")

with col3:
    st.metric("**PAR60**", f"{kpi['PAR60'].iloc[0]}%")

with col4:
    st.metric("**Avg Loan Size**", f"KES {kpi['avg_loan_size'].iloc[0]:,}")

st.divider()

# ==================== TOGGLE BUTTONS ====================
col_a, col_b, col_c = st.columns(3)

if 'show_vintage' not in st.session_state:
    st.session_state.show_vintage = True
if 'show_risk' not in st.session_state:
    st.session_state.show_risk = False
if 'show_device' not in st.session_state:
    st.session_state.show_device = False

with col_a:
    if st.button("📈 Vintage Analysis", use_container_width=True):
        st.session_state.show_vintage = not st.session_state.show_vintage

with col_b:
    if st.button("🔍 Risk Segmentation", use_container_width=True):
        st.session_state.show_risk = not st.session_state.show_risk

with col_c:
    if st.button("📱 Device Performance", use_container_width=True):
        st.session_state.show_device = not st.session_state.show_device

# ==================== VINTAGE ANALYSIS ====================
if st.session_state.show_vintage:
    with st.expander("📈 Vintage Analysis", expanded=True):
        vintage_query = """
        SELECT 
            DATE_FORMAT(disbursed_date, '%%Y-%%m') as vintage_month,
            COUNT(loan_id) as loans,
            ROUND(SUM(principal), 0) as disbursed,
            ROUND(AVG(CASE WHEN max_days_late >= 30 THEN 1 ELSE 0 END)*100, 2) as par30_pct
        FROM int_loans_repayments
        GROUP BY vintage_month
        ORDER BY vintage_month
        """
        vintage_df = conn.query(vintage_query)
        
        col1, col2 = st.columns(2)
        with col1:
            fig1 = px.line(vintage_df, x='vintage_month', y='par30_pct', markers=True,
                           title="PAR30 Trend by Vintage Month")
            fig1.update_layout(xaxis_tickangle=-45)
            st.plotly_chart(fig1, use_container_width=True)
        
        with col2:
            fig2 = px.bar(vintage_df, x='vintage_month', y='disbursed',
                          title="Disbursed Amount by Vintage")
            fig2.update_layout(xaxis_tickangle=-45)
            st.plotly_chart(fig2, use_container_width=True)

# ==================== RISK SEGMENTATION ====================
if st.session_state.show_risk:
    with st.expander("🔍 Risk Segmentation", expanded=True):
        seg_query = """
        SELECT 
            c.location, c.income_band,
            COUNT(l.loan_id) as loan_count,
            ROUND(AVG(CASE WHEN r.max_days_late >= 30 THEN 1 ELSE 0 END)*100, 2) as par30_rate
        FROM stg_loans l
        JOIN stg_customers c ON l.customer_id = c.customer_id
        JOIN int_loans_repayments r ON l.loan_id = r.loan_id
        GROUP BY c.location, c.income_band
        """
        segments = conn.query(seg_query)
        st.dataframe(segments.sort_values('par30_rate', ascending=False), use_container_width=True)

# ==================== DEVICE PERFORMANCE ====================
if st.session_state.show_device:
    with st.expander("📱 Device Performance Analysis", expanded=True):
        device_query = """
        SELECT 
            device_model,
            COUNT(loan_id) as loan_count,
            ROUND(AVG(CASE WHEN max_days_late >= 30 THEN 1 ELSE 0 END)*100, 2) as par30_rate
        FROM int_loans_repayments
        GROUP BY device_model
        ORDER BY par30_rate DESC
        """
        device_perf = conn.query(device_query)
        st.dataframe(device_perf, use_container_width=True)

# ==================== FOOTER ====================
st.divider()
st.caption("**Developed by Aklilu Abera** | Data Analyst")
st.caption("MoPhones Credit Portfolio Health Monitor • Built with MySQL • dbt • Streamlit")

Writing streamlit_app.py
